In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
from bs4 import BeautifulSoup
import torch

d:\medved_anikin_1710\find_meaning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

HF_MODEL = os.getenv('HF_MODEL')
QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
QDRANT_COLLECTION = os.getenv("QDRANT_COLLECTION", "dialogs")

In [3]:
qdrant = QdrantClient(url=QDRANT_URL)

In [4]:
embedder = SentenceTransformer("intfloat/e5-small")

In [7]:
vec = embedder.encode('налоги').tolist()

In [10]:
results = qdrant.query_points(
    collection_name='embedded_data',
    using='dialog',
    query=vec,
    limit=3,
)

In [16]:
def souped(html_text):
    soup = BeautifulSoup(html_text, 'html.parser')
    souped = soup.get_text(strip=True)
    return souped

In [15]:
results.points[0].payload['txt']

'<span class=participant_1>Пользователь 1: Привет</span><br /><span class=participant_2>Пользователь 2: Привет</span><br /><span class=participant_1>Пользователь 1: Как прошёл день?</span><br /><span class=participant_2>Пользователь 2: Довольно насыщенно</span><br /><span class=participant_1>Пользователь 1: Куча работы?</span><br /><span class=participant_2>Пользователь 2: Как у тебя？</span><br /><span class=participant_1>Пользователь 1: О, все кругом идиоты, правил не знают, пассажиры через<br />одного были пьяные. Но хотя бы удалось неплохо заработать<br />- будет чем оплатить кредит за машину</span><br /><span class=participant_2>Пользователь 2: Я все по дому крутилась, затем в дет. сад за малышкой<br />и т. п.</span><br /><span class=participant_1>Пользователь 1: Так ты домохозяйка? Муж, наверное, хорошо зарабатывает<br />?</span><br /><span class=participant_2>Пользователь 2: Есть такое....А ты таксист？</span><br /><span class=participant_1>Пользователь 1: Да .... У меня и професс

In [20]:
print("\n".join([souped(p.payload["txt"]) for p in results.points]))

Пользователь 1: ПриветПользователь 2: ПриветПользователь 1: Как прошёл день?Пользователь 2: Довольно насыщенноПользователь 1: Куча работы?Пользователь 2: Как у тебя？Пользователь 1: О, все кругом идиоты, правил не знают, пассажиры черезодного были пьяные. Но хотя бы удалось неплохо заработать- будет чем оплатить кредит за машинуПользователь 2: Я все по дому крутилась, затем в дет. сад за малышкойи т. п.Пользователь 1: Так ты домохозяйка? Муж, наверное, хорошо зарабатывает?Пользователь 2: Есть такое....А ты таксист？Пользователь 1: Да .... У меня и профессиональное заболевание имеется. Геморрой. Ты не знаешь какого- нибудь хорошего врачапо этой части?Пользователь 2: К сожалению нет, но могу навести справки у мужа, ондантист и может что нибудь знать в этой сфере.Пользователь 1: Да, я был бы очень благодарен. А у тебя есть какое- то хобби?Пользователь 2: Я люблю дорогу, путешествия, горы .... так же люблюизобретать всякие нужные и не очень нужные вещиПользователь 2: А ты что любишь？Пользова

In [6]:
from qdrant_client.models import QueryRequest, Filter, FieldCondition, MatchValue

client = QdrantClient("localhost", port=6333)

def search_with_query_points(collection_name, queries, limit=10):
    """
    Использование QueryRequest для поиска по нескольким векторам
    """
    query_request = QueryRequest(
        collection_name=collection_name,
        query=queries,  # Может быть одним запросом или списком
        limit=limit,
        with_payload=True,
        with_vectors=False
    )
    
    return client.query_points(query_request)

# Пример 1: Поиск по одному вектору
single_query = {
    "using": "dialog",  # Указываем, по какому вектору искать
    "vector": vec  # Вектор запроса
}

results = search_with_query_points(
    collection_name="embedded_data",
    queries=single_query,
    limit=5
)

ValidationError: 46 validation errors for QueryRequest
query.list[float]
  Input should be a valid list [type=list_type, input_value={'using': 'dialog', 'vect..., 0.046176113188266754]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/list_type
query.SparseVector.indices
  Field required [type=missing, input_value={'using': 'dialog', 'vect..., 0.046176113188266754]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
query.SparseVector.values
  Field required [type=missing, input_value={'using': 'dialog', 'vect..., 0.046176113188266754]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
query.SparseVector.using
  Extra inputs are not permitted [type=extra_forbidden, input_value='dialog', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/extra_forbidden
query.SparseVector.vector
  Extra inputs are not permitted [type=extra_forbi

In [ ]:
def souped(html_text):
    soup = BeautifulSoup(html_text, 'html.parser')
    souped = soup.get_text(strip=True)
    return souped

def find_top_n_contexts(question, n, target_vector='dialog'):
    vec = embedder.encode(question).tolist()
    results = qdrant.query_points(collection_name=QDRANT_COLLECTION, 
                            query_vector=(vec, target_vector,), 
                            limit=n)
    context = "\n".join([souped(p.payload["txt"]) for p in results])
    return context
    

In [ ]:
find_top_n_contexts('налоги', 5)

AssertionError: Unknown arguments: ['query_vector']